In [238]:
## NEED TO SPECIFY WHICH VERSION OF PYTHON TO USE 3.15.1

# What we are building

TBC

## 1. Setup & Preflight
Install Dependancies (from requirements.txt) <br>
Import <br>
Setup Logging <br>
Config <br>
Preflight Check <br>

### 1.1 Install Depedancies

In [239]:
# Install dependencies from requirements.txt
%pip install --quiet -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [240]:
# Automate a requirements.txt file using the notebook and versions

from importlib.metadata import distributions

# Write the installed packages and versions to requirements.txt
with open("requirements.txt", "w", encoding="utf-8") as f:
    for distribution in sorted(
        distributions(),
        key=lambda dist: (dist.metadata.get("Name") or "").lower()
    ):
        name = distribution.metadata.get("Name")
        if name:
            f.write(f"{name}=={distribution.version}\n")

In [241]:
# Import libraries

from urllib.parse import urlparse
import requests
import pandas as pd
from bs4 import BeautifulSoup
import time
import json
import logging
import requests
from urllib.robotparser import RobotFileParser
from pathlib import Path
import nltk


### 1.2 Setup Logging

In [242]:
# Set up logging and log to both console and file

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler("scraper.log"),
        logging.StreamHandler(),
    ],
    force=True)

logging.info("Logging started.")

2026-11-16 12:28:24,731 [INFO] Logging started.


### 1.3 Configuration

In [243]:
### SCRAPER CONFIGURATION ###
BASE_URL = "https://forums.moneysavingexpert.com/"
DELAY = 2
HEADER = {"User-Agent": "BPP-DataScience-Bot/1.0 (educational project)"}

### FORUM CONFIGURATION ###
FORUM_URL = "https://forums.moneysavingexpert.com/categories/benefits-tax-credits"
NUMBER_OF_THREADS_TO_SCRAPE = 30

### CACHING CONFIGURATION ###
DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
CACHE_PATH = DATA_DIR / "mse_posts.jsonl"
CACHE_META = DATA_DIR / "mse_posts.meta.json"
REFRESH_CACHE = False         # set True to collect afresh from the live API


### 1.4 Pre-flight check

Check the environment to ensure XYZ <br>
Add anything to this which requires libraries and access

Source: BPP University (2026) ‘Consolidation 1: Blue Sky Weekly Digest.ipynb’, Unstructured Data. Available at: BPP (Accessed: 16 November 2026).

In [244]:
def preflight() -> dict:
    """Check what this environment can do, and add to logging.
    If there is no network access and no cache, raise an error and log a critical message.
    
    Returns:
        dict: A dictionary of the environment status
    """
    status = {}

    # Check if BeautifulSoup is installed, if not, log a warning and set status to False
    try:
        from bs4 import BeautifulSoup 
        status["BeautifulSoup installed"] = True
        logging.info(f"BeautifulSoup installed: {status['BeautifulSoup installed']}")
    except ImportError:
        status["BeautifulSoup installed"] = False
        logging.warning(f"BeautifulSoup installed: {status['BeautifulSoup installed']}")
    
    # Check if the BASE_URL is reachable, if not, log a warning and set status to False
    try:
        r = requests.get(BASE_URL, headers=HEADER, timeout=10)
        status[f"{BASE_URL} reachable"] = r.status_code == 200
        logging.info(f"{BASE_URL} reachable: {status[f'{BASE_URL} reachable']}")
    except requests.RequestException:
        status[f"{BASE_URL} reachable"] = False
        logging.warning(f"{BASE_URL} reachable: {status[f'{BASE_URL} reachable']}")
    
    # Check if the cache file exists, if not, log a warning and set status to False
    status["cached posts available"] = CACHE_PATH.exists()
    if status["cached posts available"]:
        logging.info(f"Cached posts available: {status['cached posts available']}")
    else:
        logging.warning(f"Cached posts available: {status['cached posts available']}")

    # Check if the WordNet lemmatiser is available, if not, log a warning and set status to False
    # try:
    #     import nltk
    #     nltk.download("wordnet", quiet=True)
    #     nltk.download("omw-1.4", quiet=True)
    #     from nltk.stem import WordNetLemmatizer
    #     WordNetLemmatizer().lemmatize("models")
    #     status["WordNet lemmatiser"] = True
    #     logging.info(f"WordNet lemmatiser available: {status['WordNet lemmatiser']}")
    # except Exception:
    #     status["WordNet lemmatiser"] = False
    #     logging.warning(f"WordNet lemmatiser available: {status['WordNet lemmatiser']}")    
    
    # If there is no network access and no cache, log a critical message and raise an error
    if not (status[f"{BASE_URL} reachable"] or status["cached posts available"]):
        # throw an error if there is no network access and no cache
        logging.critical("\n⚠️  No network access and no cache. Source the cached data file "
                      f"and place it at {CACHE_PATH}.") 
        raise RuntimeError("No network access and no cache available.")
        
    return status

ENV = preflight()
HAVE_BEATIFULSOUP = ENV["BeautifulSoup installed"]
HAVE_CACHE = ENV["cached posts available"]
ONLINE = ENV[f"{BASE_URL} reachable"]
# HAVE_LEMMATISER = ENV["WordNet lemmatiser"]

2026-11-16 12:28:24,749 [INFO] BeautifulSoup installed: True
2026-11-16 12:28:24,891 [INFO] https://forums.moneysavingexpert.com/ reachable: True
2026-11-16 12:28:24,891 [WARNING] Cached posts available: False


## 2. Web Scraping 

Create RepectfulScraper <br>
Source: BPP University (2026) ‘Topic 3: Web Scraping.ipynb’, Unstructured Data. Available at: BPP (Accessed: 30 October 2026).


In [247]:
# First we should create a ethical scraper which respects the robots.txt file and the terms of service of the website.

class RespectfulScraper:
    """
    A web scraper that respects robots.txt rules and rate limits.

    This class automatically:
    - Parses robots.txt for the target domain
    - Enforces crawl delays between requests
    - Identifies itself with a descriptive User-Agent string
    - Logs all activity for transparency and debugging
    """

    def __init__(self, base_url: str, default_delay: int = 3):
        self.base_url = base_url.rstrip("/")

        # Create a session for connection reuse and consistent headers
        self.session = requests.Session()
        self.session.headers.update({
            "User-Agent": "BPP-DataScience-Bot/1.0 (educational project)"
        })

        # Set up logging
        self.logger = logging.getLogger(f"scraper.{urlparse(base_url).hostname}")
        self.logger.setLevel(logging.INFO)

          # Parse robots.txt (fetched with our own session/User-Agent)
        robots_url = f"{self.base_url}/robots.txt"
        self.rp = RobotFileParser()
        self.rp.set_url(robots_url)

        resp = self.session.get(robots_url, timeout=10)
        
        if resp.status_code == 200:
            self.rp.parse(resp.text.splitlines())
            self.logger.info(f"Response {resp.status_code}: Successfully parsed robots.txt")
        elif resp.status_code in (401, 403):
            self.rp.disallow_all = True
            self.logger.warning(f"robots.txt returned {resp.status_code}; treating site as off-limits")
        elif 400 <= resp.status_code < 500:
            self.rp.allow_all = True
            self.logger.info(f"No robots.txt ({resp.status_code}); everything allowed")
        else:
            raise RuntimeError(f"Couldn't fetch robots.txt ({resp.status_code}); not crawling")

        delay = self.rp.crawl_delay(self.session.headers["User-Agent"])
        self.crawl_delay = float(delay) if delay else default_delay

    def can_fetch(self, url: str) -> bool:
        """Check if a URL is allowed by robots.txt."""
        return self.rp.can_fetch(self.session.headers["User-Agent"], url)

    def get(self, url: str) -> Optional[BeautifulSoup]:
        """
        Fetch and parse a page, respecting robots.txt and crawl delays.

        Returns a BeautifulSoup object if successful, None otherwise.
        """
        if not self.can_fetch(url):
            self.logger.warning(f"Blocked by robots.txt: {url}")
            return None

        try:
            self.logger.info(f"Fetching {url}")
            response = self.session.get(url, timeout=10)
            response.raise_for_status()

            # Respect crawl delay
            time.sleep(self.crawl_delay)

            return BeautifulSoup(response.text, "html.parser")

        except requests.exceptions.RequestException as e:
            self.logger.error(f"Error fetching {url}: {e}")
            return None

In [246]:
mse_scraper = RespectfulScraper(base_url=BASE_URL, default_delay=DELAY, headers=HEADER)

TypeError: RespectfulScraper.__init__() got an unexpected keyword argument 'headers'

In [ ]:
# 1. Create scraper and check the robots.txt file etc
# 2. Scrpae the url provided and get the thread data (url, pinned, announcement, countViews, countComments)
    ## Count the number of records if less than the number specified in the config, then scrape the next page until the number of records is reached or there are no more pages to scrape.
    
def scrape_forum(scraper, forum_url: str, max_threads: int = 10) -> pd.DataFrame:
    """Scrape a forum for the specified number of threads and return a DataFrame with the thread data.

    Args:
        scraper (_type_): _description_
        forum_url (str): _description_
        max_threads (int, optional): _description_. Defaults to 10.

    Returns:
        pd.DataFrame: _description_
    """
    
    count_of_threads = 0
    
    # Extract the number of posts from the current page
    thread_data = []
    
    url_list = [forum_url]
    
    for url in url_list:
        if count_of_threads >= max_threads:
            break
        
        else:
            soup = scraper.get(url)
            for dataprops in soup.find_all("div", attrs={"data-props": True}):
                props = json.loads(dataprops["data-props"])

                for thread in props.get("discussions", []):
                    if count_of_threads >= max_threads:
                        break
                    else:
                        if thread.get("pinned") is not True and thread.get("announce") is not True:
                            thread_data.append({
                                "url": thread.get("url"),
                                "pinned": thread.get("pinned"),
                                "announcement": thread.get("announce"),
                                "count_views": thread.get("countViews"),
                                "count_comments": thread.get("countComments"),
                            })
                            count_of_threads += 1
            if count_of_threads >= max_threads:
                break
            else: 
                next_page_link = soup.find("div", class_="PageControls Top").find("a", class_="Next")["href"]
                url_list.append(next_page_link)
                
    return pd.DataFrame(thread_data)
    

In [ ]:
threads_df = scrape_forum(mse_scraper, FORUM_URL, max_threads=NUMBER_OF_THREADS_TO_SCRAPE)

In [ ]:
threads_df

In [ ]:
def scrape_threads(scraper, threads_df: pd.DataFrame) -> pd.DataFrame:
    
    threads_df = threads_df.copy()
    
    for url in threads_df["url"]:
        soup = scraper.get(url)
        # Get the first post content from the thread page
        first_post_content = soup.find("div", class_="Message userContent")
        threads_df.loc[threads_df["url"] == url, "first_post_content"] = str(first_post_content)
    return threads_df

In [ ]:
posts = scrape_threads(mse_scraper, threads_df)


In [ ]:
posts